# Convolutional Neural Networks

## Project: Write an Algorithm for Landmark Classification


### Transfer learning

In the previous notebook we have trained our own CNN and we got a certain performance. Let's see how hard it is to match that performance with transfer learning.

---
## <img src="static_images/icons/noun-advance-2109145.png" alt=">" style="width:50px"/> Step 0: Setting up

The following cells make sure that your environment is setup correctly and check that your GPU is available and ready to go. You have to execute them every time you restart your notebook.

In [1]:
# Install requirements
#!pip install -r requirements.txt | grep -v "already satisfied"

In [2]:
# Automatically reload modules before executing user code
%load_ext autoreload 
%autoreload 2 

In [3]:
from src.helpers import setup_env

# If running locally, this will download dataset (make sure you have at 
# least 2 Gb of space on your hard drive)
setup_env()

GPU available
Dataset already downloaded. If you need to re-download, please delete the directory landmark_images
Reusing cached mean and std


/workspace/code/project/src/helpers.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to pass `weights_only=True` or `weights_only=False` to set an explicit value and suppress this warning. This will be the default behavior in future versions. If you wish to keep the old behavior (not checking the saved weights), you can do this by passing a command to `torch.load` like so:
  d = torch.load(cache_file)


---
## <img src="static_images/icons/noun-advance-2109145.png" alt=">" style="width:50px"/> Step 1: Create transfer learning architecture

Open the file `src/transfer.py` and complete the `get_model_transfer_learning` function. When you are done, execute this test:

In [4]:
!pytest -vv src/transfer.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3.12
cachedir: .pytest_cache
rootdir: /workspace/code/project
configfile: pytest.ini
plugins: anyio-4.12.1
collected 1 item                                                               

src/transfer.py::test_get_model_transfer_learning PASSED                 [100%]

=============================== warnings summary ===============================
src/transfer.py::test_get_model_transfer_learning
  /workspace/code/project/src/helpers.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to pass `weights_only=True` or `weights_only=False` to set an explicit value and suppress this warning. This will be the default behavior in future versions. If you wish to keep the old behavior (not checking the saved weights), you c

---
## <img src="static_images/icons/noun-advance-2109145.png" alt=">" style="width:50px"/> Step 2: Train, validation and test

Let's train our transfer learning model! Let's start defining the hyperparameters:

In [1]:
batch_size = 32  # size of the minibatch for stochastic gradient descent (or Adam)
valid_size = 0.2  # fraction of the training data to reserve for validation
num_epochs = 10  # number of epochs for training
num_classes = 50  # number of classes. Do not change this
learning_rate = 0.001  # Learning rate for SGD (or Adam)
opt = 'adam'      # optimizer. 'sgd' or 'adam'
weight_decay = 0.0001 # regularization. Increase this to combat overfitting

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

In [ ]:
from src.data import get_data_loaders
from src.optimization import get_optimizer, get_loss
from src.train import optimize
from src.transfer import get_model_transfer_learning

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_transfer = get_model_transfer_learning(
    "resnet18",
    n_classes=num_classes
).to(device)

data_loaders = get_data_loaders(
    batch_size=batch_size,
    valid_size=valid_size,
    num_workers=2
)

optimizer = get_optimizer(
    model_transfer,
    learning_rate=learning_rate,
    optimizer=opt,
    weight_decay=weight_decay,
)

loss = get_loss()

optimize(
    data_loaders,
    model_transfer,
    optimizer,
    loss,
    n_epochs=num_epochs,
    save_path="checkpoints/model_transfer.pt",
    interactive_tracking=False
)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. Please use `weights=ResNet18_Weights.IMAGENET1K_V1` or the appropriate weights enum instead.
  warnings.warn(msg)


Reusing cached mean and std
Dataset mean: tensor([0.2718, 0.2709, 0.2675]), std: tensor([0.3202, 0.3338, 0.3576])


Validating: 100%|███████████████████████████████| 32/32 [00:06<00:00,  4.91it/s]

Epoch: 1 	Training Loss: 3.206943 	Validation Loss: 2.327318
New minimum validation loss: 2.327318. Saving model ...



Validating: 100%|███████████████████████████████| 32/32 [00:05<00:00,  6.04it/s]


Epoch: 2 	Training Loss: 2.117827 	Validation Loss: 1.770992
New minimum validation loss: 1.770992. Saving model ...



Validating: 100%|███████████████████████████████| 32/32 [00:05<00:00,  6.17it/s]

Epoch: 3 	Training Loss: 1.712660 	Validation Loss: 1.532054
New minimum validation loss: 1.532054. Saving model ...


<img src="static_images/icons/noun-question-mark-869751.png" alt="?" style="width:25px"/> __Question:__ Outline the steps you took to get to your final CNN architecture and your reasoning about the choices you made.

<img src="static_images/icons/noun-answer-3361020.png" alt=">" style="width:25px"/>  __Answer:__ 

For the transfer learning model, I used a pre-trained ResNet18 architecture as the backbone and replaced its final fully connected layer with a new linear layer that outputs the required number of landmark classes.

The first step was to choose a pre-trained convolutional neural network. I selected ResNet18 because it is a well-known CNN architecture that performs well on image classification tasks while still being lightweight enough for rapid training.

The second step was to load the ResNet18 model with pre-trained weights. Using a pre-trained model is useful because the network has already learned general image features from a large image dataset (ImageNet).

The third step was to freeze the parameters of the pre-trained backbone. By freezing the convolutional layers, the model keeps the useful visual representations learned during pre-training and only trains the newly added classification layers.

The fourth step was to replace the original final classification layer with a new linear layer. The input size of this layer matches the number of features produced by the ResNet18 backbone, and the output size matches the number of landmark classes (50).

This architecture is suitable for the current problem because landmark classification is an image classification task, and ResNet18 is designed to learn hierarchical visual features from images.

---
## <img src="static_images/icons/noun-advance-2109145.png" alt=">" style="width:50px"/> Step 3: Test the Model

Try out your model on the test dataset of landmark images. Use the code cell below to calculate and print the test loss and accuracy. Ensure that your test accuracy is greater than 60% and matches the accuracy of the model trained from scratch.

In [8]:
import torch
from src.train import one_epoch_test
from src.transfer import get_model_transfer_learning

model_transfer = get_model_transfer_learning("resnet18", n_classes=num_classes)
model_transfer.load_state_dict(torch.load('checkpoints/model_transfer.pt'))

one_epoch_test(data_loaders['test'], model_transfer, loss)

/tmp/ipykernel_501/294021750.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to pass `weights_only=True` or `weights_only=False` to set an explicit value and suppress this warning. This will be the default behavior in future versions. If you wish to keep the old behavior (not checking the saved weights), you can do this by passing a command to `torch.load` like so:
  model_transfer.load_state_dict(torch.load('checkpoints/model_transfer.pt'))
Testing: 100%|██████████████████████████████████| 40/40 [00:12<00:00,  3.31it/s]


Test Loss: 1.163516

Test Accuracy: 71% (889/1250)


1.1635156372562052

---
## <img src="static_images/icons/noun-advance-2109145.png" alt=">" style="width:50px"/> Step 4: Export using torchscript

Now, just like we did with our original model, we export the best fit model using torchscript so that it can be used in our application:

In [9]:
from src.predictor import Predictor
from src.helpers import compute_mean_and_std

class_names = data_loaders["train"].dataset.classes
model_transfer = model_transfer.cpu()
model_transfer.load_state_dict(
    torch.load("checkpoints/model_transfer.pt", map_location="cpu")
)

mean, std = compute_mean_and_std()
predictor = Predictor(model_transfer, class_names, mean, std).cpu()

scripted_predictor = torch.jit.script(predictor)
scripted_predictor.save("checkpoints/transfer_exported.pt")

/tmp/ipykernel_501/1260189282.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to pass `weights_only=True` or `weights_only=False` to set an explicit value and suppress this warning. This will be the default behavior in future versions. If you wish to keep the old behavior (not checking the saved weights), you can do this by passing a command to `torch.load` like so:
  torch.load("checkpoints/model_transfer.pt", map_location="cpu")


Reusing cached mean and std


In [10]:
import torch
from src.predictor import predictor_test
from src.helpers import plot_confusion_matrix

model_reloaded = torch.jit.load("checkpoints/transfer_exported.pt")
pred, truth = predictor_test(data_loaders['test'], model_reloaded)
plot_confusion_matrix(pred, truth)

100%|███████████████████████████████████████| 1250/1250 [04:41<00:00,  4.44it/s]


Accuracy: 0.7096
